# 02 – Credit Lifecycle

The low-level reserve-then-deduct pattern via `PostgresStore`:

1. **Add** — deposit credits for a user
2. **Reserve** — hold credits before an expensive operation
3. **Deduct** — consume the reservation
4. **Check** — verify balance
5. **Refund** — reverse a deduction

In [ ]:
from datetime import datetime, timedelta
from ducto.interface.postgres import PostgresStore
from ducto.manager import CreditManager
from ducto.engine import PricingEngine
from ducto.metrics import UsageMetrics, ToolCall
from ducto.interface.models import (
    PricingConfigData, PricingConfigV2, PlanDefinition,
    CreditMetadata,
)
from shared import start_postgres_store, cleanup

store, pgdata = start_postgres_store()
import uuid
print("✔ PostgresStore ready.")

### Add credits

In [ ]:
user = str(uuid.uuid4())
r = store.add_credits(user, 10_000, type="signup_bonus")
print(f"  Tx:        {r.transaction_id}")
print(f"  Balance:   {r.new_balance}")

### Reserve → deduct (two-phase commit)

In [ ]:
res = store.reserve_credits(user, 2_000, operation_type="model_inference")
print(f"  Reservation: {res.reservation_id}")
print(f"  Balance:     {res.balance}")

ded = store.deduct_credits(user, res.reservation_id, 2_000)
print(f"  Deduction:   {ded.transaction_id}")
print(f"  Balance aft: {ded.balance_after}")

bal = store.get_balance(user)
print(f"  Final:       {bal.balance}")
assert bal.balance == 8_000

### Refund a deduction

In [ ]:
ref = store.refund_credits(ded.transaction_id, amount=2_000, reason="test")
print(f"  Refund tx:   {ref.refund_transaction_id}")
print(f"  New balance: {ref.new_balance}")
assert ref.new_balance == 10_000

In [ ]:
cleanup(pgdata)